# EoMT LoRA Fine-Tuning (10 epochs) + OOD Evaluation + Logit Norm Loss

This notebook fine-tunes EoMT with LoRA for 10 epochs and then runs MSP, MaxLogit, and MaxEntropy on anomaly datasets.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Set your paths. `DATA_DIR` must contain the Cityscapes zip files (`leftImg8bit_trainvaltest.zip` and `gtFine_trainvaltest.zip`).

In [ ]:
import os
from pathlib import Path

CODE_DIR = "/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject"  # TODO: change
DATA_DIR = "/content/drive/MyDrive/datasets"  # TODO: change
OOD_INPUT_GLOB = "/content/drive/MyDrive/anomaly/RoadObsticle21/images/*.webp"  # TODO: change

CKPT_PATH = os.path.join(CODE_DIR, "eomt/checkpoints/eomt_cityscapes.bin")

In [ ]:
os.chdir(CODE_DIR)
print("Working dir:", os.getcwd())

Install dependencies. This keeps the Colab-provided torch/torchvision versions.

In [ ]:
import pathlib

reqs = pathlib.Path("eomt/requirements.txt").read_text().splitlines()
filtered = [r for r in reqs if not r.startswith("torch==") and not r.startswith("torchvision==")]
pathlib.Path("/tmp/eomt_requirements.txt").write_text("\n".join(filtered))

!pip install -r /tmp/eomt_requirements.txt

Create a LoRA fine-tuning config (10 epochs). Adjust batch size if you hit OOM.

In [ ]:
import textwrap

lora_config = f'''
trainer:
  max_epochs: 10
  default_root_dir: "outputs"
  logger: false
  log_every_n_steps: 1
  enable_progress_bar: true
  enable_checkpointing: true
  callbacks:
    - class_path: lightning.pytorch.callbacks.ModelCheckpoint
      init_args:
        dirpath: "outputs/checkpoints"
        save_last: true
        save_top_k: 0
model:
  init_args:
    ckpt_path: "{CKPT_PATH}"
    lora_enabled: true
    lora_r: 8
    lora_alpha: 16.0
    lora_dropout: 0.05
    lora_target_modules: ["qkv", "proj"]
    lora_train_bias: "none"
    lora_trainable_modules: ["class_head", "mask_head", "q"]
data:
  init_args:
    path: "{DATA_DIR}"
    batch_size: 1
    num_workers: 2
'''

Path("lora_finetune.yaml").write_text(textwrap.dedent(lora_config))
print(Path("lora_finetune.yaml").read_text())

Run LoRA fine-tuning for 10 epochs with logit norm loss

In [ ]:
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_SILENT"] = "true"

!python eomt/main.py fit \
  -c eomt/configs/dinov2/cityscapes/semantic/eomt_base_640_logitnorm.yaml \
  -c lora_finetune.yaml \
  --compile_disabled

In [ ]:
import glob

ckpt_candidates = sorted(
    glob.glob("outputs/checkpoints/*.ckpt"),
    key=os.path.getmtime,
)
if not ckpt_candidates:
    ckpt_candidates = sorted(
        glob.glob("outputs/**/checkpoints/*.ckpt", recursive=True),
        key=os.path.getmtime,
    )
if not ckpt_candidates:
    raise FileNotFoundError("No checkpoints found under outputs/**/checkpoints")
FINETUNED_CKPT = ckpt_candidates[-1]
print("Using checkpoint:", FINETUNED_CKPT)

Evaluate MSP, MaxLogit, and MaxEntropy. Update `OOD_INPUT_GLOB` to point at each dataset split you want to test.

In [ ]:
import subprocess

methods = ["msp", "maxlogit", "maxentropy"]
for method in methods:
    cmd = [
        "python",
        "eomt/evalAnomaly.py",
        "--input",
        OOD_INPUT_GLOB,
        "--ckpt",
        FINETUNED_CKPT,
        "--method",
        method,
        "--lora",
    ]
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()

**High-resolution anomaly map visualization for all supported datasets.**

This cell defines and runs a function to generate and save high-res anomaly heatmaps and overlays for several datasets, using the trained model. The output images are saved in a dedicated folder for each dataset.

In [ ]:
# --- HIGH-RES VISUALIZATION CELL (ALL DATASETS) ---
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
from pathlib import Path
import random
import numpy as np
import os
import scipy.ndimage
import cv2  # Use OpenCV for high-quality resizing

def visualize_thesis_high_res(model, dataset_name, num_samples=3):
    """
    Generate and save high-resolution anomaly heatmaps and overlays for a given dataset.
    Args:
        model: Trained segmentation model.
        dataset_name: Name of the dataset to visualize.
        num_samples: Number of random samples to visualize.
    """
    model.eval()
    temp = 0.1  # Natural temperature for clean visualization

    val_root = Path(CONFIG['val_root'])

    # --- UPDATED FOLDER MAPPING (ALL DATASETS) ---
    folder_map = {
        "Fishyscapes":          "fs_static",
        "Fishyscapes (Static)": "fs_static",
        "RoadObsticle":         "RoadObsticle21",
        "RoadObsticle21":       "RoadObsticle21",
        "RoadAnomaly":          "RoadAnomaly",
        "RoadAnomaly21":        "RoadAnomaly21",     # <--- ADDED
        "Lost & Found":         "FS_LostFound_full", # <--- ADDED
        "FS_LostFound_full":    "FS_LostFound_full"  # <--- ADDED (Alias)
    }

    # Resolve actual folder name
    actual_folder = folder_map.get(dataset_name, dataset_name)
    ds_path = val_root / actual_folder

    print(f"\n📸 Generating High-Res for: {dataset_name} (Folder: {actual_folder})...")

    # Search for images
    img_list = []
    if ds_path.exists():
        for root, dirs, files in os.walk(ds_path):
            if "labels_masks" in root or "gtFine" in root:
                continue
            for file in files:
                if file.lower().endswith((".jpg", ".png", ".webp")):
                    if "mask" not in file.lower() and "label" not in file.lower():
                        img_list.append(os.path.join(root, file))

    if not img_list:
        print(f" No images found in {ds_path}")
        return

    # Random sample selection
    samples = random.sample(img_list, min(len(img_list), num_samples))

    # Dedicated output folder
    save_dir = os.path.join(CONFIG['output_dir'], 'Thesis_HighRes_Images', actual_folder)
    os.makedirs(save_dir, exist_ok=True)

    for i, img_path in enumerate(samples):
        try:
            # 1. Load original image and save true dimensions
            raw_img = Image.open(img_path).convert("RGB")
            original_w, original_h = raw_img.size

            # 2. Preprocessing (Resize to 1024x1024 for the model)
            img_t = transforms.Compose([
                transforms.Resize((1024, 1024)),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
            ])(raw_img).unsqueeze(0).to(CONFIG['device'])

            # 3. Inference
            with torch.no_grad():
                m_logits, c_logits = model.network(img_t)
                c_probs = (c_logits[-1] / temp).softmax(-1)[..., :-1]
                m_probs = m_logits[-1].sigmoid()
                semantic_probs = torch.einsum("bqhw,bqc->bchw", m_probs, c_probs)

                # Internal upsampling
                if semantic_probs.shape[-1] != 1024:
                    semantic_probs = F.interpolate(semantic_probs, size=(1024, 1024), mode='bilinear', align_corners=False)

                conf_map, _ = torch.max(semantic_probs.squeeze(0), dim=0)
                anomaly_map_small = 1.0 - conf_map.cpu().numpy()

            # 4. SMART RESIZE (From 1024 to original dimensions)
            anomaly_map_full = cv2.resize(anomaly_map_small, (original_w, original_h), interpolation=cv2.INTER_LINEAR)

            # Light smoothing
            anomaly_map_full = scipy.ndimage.gaussian_filter(anomaly_map_full, sigma=2.0)

            # Relative normalization
            v_min, v_max = anomaly_map_full.min(), anomaly_map_full.max()
            if v_max - v_min < 0.001:
                v_max = 1.0

            # 5. Plot
            fig, axes = plt.subplots(1, 3, figsize=(18, 6))

            # Input
            axes[0].imshow(raw_img)
            axes[0].set_title(f"Original Input ({original_w}x{original_h})")
            axes[0].axis('off')

            # Heatmap
            im = axes[1].imshow(anomaly_map_full, cmap='inferno', vmin=v_min, vmax=v_max)
            axes[1].set_title("Anomaly Heatmap")
            axes[1].axis('off')
            plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

            # Overlay
            axes[2].imshow(raw_img)
            axes[2].imshow(anomaly_map_full, cmap='jet', alpha=0.45, vmin=v_min, vmax=v_max)
            axes[2].set_title("Overlay (High Res)")
            axes[2].axis('off')

            out_file = os.path.join(save_dir, f"highres_{i}.png")
            plt.savefig(out_file, bbox_inches='tight', dpi=150)
            plt.show()
            print(f" Saved: {out_file}")

        except Exception as e:
            print(f" Error on {img_path}: {e}")

# --- FULL EXECUTION ---
# Generate images for ALL datasets (including newly added ones)
visualize_thesis_high_res(model, "Fishyscapes", num_samples=2)
visualize_thesis_high_res(model, "RoadObsticle21", num_samples=2)
visualize_thesis_high_res(model, "RoadAnomaly", num_samples=2)

# These now work too!
visualize_thesis_high_res(model, "RoadAnomaly21", num_samples=2)
visualize_thesis_high_res(model, "Lost & Found", num_samples=2)
